In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix, classification_report

In [ ]:
# ──── Paths ──────────────────────────────────────────────────────
TEST_TARGET       = '../data/processed/y_test.csv'
BASE_PROBS_CSV    = '../notebooks/base_probabilities.csv'
STACKED_PROBS_CSV = '../notebooks/stacked_probabilities.csv'
OPTIMAL_THRESH_TXT = '../notebooks/optimal_threshold.txt'

In [ ]:
# ──── Load everything ────────────────────────────────────────────
y_test = pd.read_csv(TEST_TARGET)['TARGET']

base_probs = pd.read_csv(BASE_PROBS_CSV)
stacked_probs = pd.read_csv(STACKED_PROBS_CSV)['stacked_prob']

with open(OPTIMAL_THRESH_TXT) as f:
    optimal_thresh = float(f.read().strip())

print(f"Using optimal threshold: {optimal_thresh:.3f}")

In [ ]:
# ──── Final metrics at optimal threshold ─────────────────────────
y_pred_opt = (stacked_probs >= optimal_thresh).astype(int)

print("\n" + "="*60)
print("FINAL EVALUATION @ OPTIMAL THRESHOLD")
print("="*60)
print(classification_report(y_test, y_pred_opt))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_opt))

In [ ]:
# ──── ROC comparison plot (final version) ────────────────────────
plt.figure(figsize=(10,8))

for col in base_probs.columns:
    fpr, tpr, _ = roc_curve(y_test, base_probs[col])
    auc_val = roc_auc_score(y_test, base_probs[col])
    plt.plot(fpr, tpr, label=f'{col} (AUC = {auc_val:.4f})', alpha=0.75)

fpr, tpr, _ = roc_curve(y_test, stacked_probs)
auc_stacked = roc_auc_score(y_test, stacked_probs)
plt.plot(fpr, tpr, label=f'Stacked Ensemble (AUC = {auc_stacked:.4f})', linewidth=3.2, color='darkred')

plt.plot([0,1], [0,1], color='gray', linestyle='--', alpha=0.5)
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curves – Final Comparison', fontsize=14)
plt.legend(loc='lower right', fontsize=11)
plt.grid(alpha=0.25)
plt.savefig('../reports/figures/06_final_roc_all_models.png', dpi=180, bbox_inches='tight')
plt.show()

In [ ]:
# ──── Confusion matrix visualization at optimal threshold ────────
cm = confusion_matrix(y_test, y_pred_opt)
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

plt.figure(figsize=(7,6))
sns.heatmap(cm_norm, annot=cm, fmt='d', cmap='Blues',
            xticklabels=['Predicted Good', 'Predicted Bad'],
            yticklabels=['Actual Good', 'Actual Bad'])
plt.title(f'Confusion Matrix @ threshold = {optimal_thresh:.3f}\n(Total cost minimized)', fontsize=13)
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.savefig('../reports/figures/06_confusion_matrix_optimal.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nEvaluation complete. Check reports/figures/ for visuals.")